# XGBoost Cross-Sectional Return Predictor

Walk-forward backtest using XGBoost on the full S&P 500 universe.
Results (charts + tables) are saved to .

In [4]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
sys.path.insert(0, '../../src')

from evaluation import WalkForwardExpandingBacktester, WalkForwardSlidingBacktester
from features.registry import FeatureStore
from models import XGBoostRankModel
import utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1  Data Loading

In [5]:
store = FeatureStore('../../data/features/yahoo_S&P_500')
store.describe()
features_df = store.load()
features_df = (features_df
 .reset_index())

In [6]:
features_df = features_df.set_index('Date')

In [7]:
FEATURES = [
    # Illiquidity
    'amihud_21',
    
    # Liquidity level (different concept: trading activity, not price impact)
    'dollar_volume_21',
    
    # Volatility — one per horizon, different info sources
    'ivol_252',
    'parkinsons_vol_63',
    'downside_dev_21',
    
    # Momentum
    'mom_252_756',
    'mom_126_252',
    
    # Extremes / Anchoring
    'max_21',
    'dist_252_high',
    
    # Market risk
    'rolling_beta',
    
    # Short-term reversal
    'mom_1_21',
]

features_df = features_df[FEATURES + ['Ticker']]

In [8]:
features_df[FEATURES] = features_df[FEATURES].groupby('Date').transform(lambda x: x.rank(pct = True))

In [9]:
_temp_df = pd.read_parquet('../../data/raw/yahoo_S&P_500/raw.parquet')
_temp_df = (_temp_df.reset_index()
          .set_index('Date')
          .sort_index())
adj = _temp_df["Adj Close"].sort_index()
fwd_ret = adj.pct_change(21, fill_method=None).shift(-21)
del(_temp_df);

In [10]:
fwd_ret = fwd_ret.stack().reset_index()
fwd_ret.columns = ['Date', 'Ticker', 'target']
fwd_ret = fwd_ret.set_index('Date')

In [11]:
fwd_ret['target'] = fwd_ret['target'].groupby('Date').transform(lambda x: x.rank(pct=True))

In [12]:
fwd_ret = fwd_ret.reset_index()
features_df = features_df.reset_index()

In [17]:
df = pd.merge(features_df, fwd_ret, on=['Date', 'Ticker'], how='left')

In [18]:
df = df.set_index('Date').sort_index()

In [19]:
days = df.index.sort_values()

In [23]:
unique_days = df.index.unique().sort_values()
t0 = unique_days[unique_days >= unique_days[0] + pd.DateOffset(years=3, days = 5)][0]
df = df[df.index > t0]

## 4  Walk-Forward Backtest

In [ ]:
MODEL_NAME = "xgboost_forwardexpanding"
RESULTS_DIR = "../../results/xgboost_forwardexpanding"

model = XGBoostRankModel(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.2,
    min_child_weight=100,
    gamma=0.5,
)

backtester = WalkForwardExpandingBacktester(
    model=model,
    initial_train_months=84,   # 7-year expanding window
    test_months=1,
    step_months=1,
    embargo_days=21,
)

"""backtester = WalkForwardSlidingBacktester(
    model=model,
    window_size=12*4,   # 7-year expanding window
    test_months=1,
    step_months=1,
    embargo_days=21,
)
"""

results = backtester.run(df, feature_cols=FEATURES, target_col="target")
print(f"Completed {len(results)} folds.")

## 5  Fold Summary

In [ ]:
fold_summary = backtester.summary(results)
print(fold_summary.to_string(index=False))

## 6  Diagnostics — IC, ICIR, L/S Spread

In [ ]:
ic_series, ls_spread = utils.diagnose(
    results,
    ic_on_every=21,
    save_dir=RESULTS_DIR,
    model_name=MODEL_NAME,
)

## 7  Feature Importance

Aggregate XGBoost feature importance across all folds (average gain).

In [ ]:
from models import XGBoostRankModel
import xgboost as xgb
import pandas as pd

# Re-run one final model on the full dataset to get stable importances
# (last fold model is already in  after the walk-forward)
all_importances = []
for r in results:
    # We stored the model per fold — use the last model for a quick estimate
    pass

# Use the model from the final fold (already fitted)
importance = model.get_feature_importance()
if importance is not None:
    fig = utils.plot_feature_importance(
        importance,
        model_name=MODEL_NAME,
        top_n=len(FEATURES),
        save_dir=RESULTS_DIR,
    )
    plt.show()
else:
    print("No feature importances available.")

## 8  Per-Fold IC Summary

In [ ]:
import matplotlib.pyplot as plt

fold_ics = []
for r in results:
    import pandas as pd
    from scipy.stats import spearmanr
    df_fold = pd.DataFrame({"pred": r["preds"], "actual": r["actuals"]}, index=r["index"])
    sampled = df_fold.index.unique().sort_values()[::21]
    df_s = df_fold[df_fold.index.isin(sampled)]
    ic = df_s.groupby("Date").apply(lambda x: x["pred"].corr(x["actual"], method="spearman")).mean()
    fold_ics.append({"fold": r["fold"], "test_start": r["test_start"], "mean_ic": ic})

fold_ic_df = pd.DataFrame(fold_ics)
print(fold_ic_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["steelblue" if v > 0 else "tomato" for v in fold_ic_df["mean_ic"]]
ax.bar(fold_ic_df["fold"], fold_ic_df["mean_ic"], color=colors)
ax.axhline(fold_ic_df["mean_ic"].mean(), color="black", linestyle="--",
           label=f'Mean = {fold_ic_df["mean_ic"].mean():.4f}')
ax.set_xlabel("Fold")
ax.set_ylabel("Mean IC")
ax.set_title("Per-Fold Mean IC — XGBoost")
ax.legend()
plt.tight_layout()

import pathlib
pathlib.Path(f"{RESULTS_DIR}/charts").mkdir(parents=True, exist_ok=True)
fig.savefig(f"{RESULTS_DIR}/charts/xgboost_per_fold_ic.png", dpi=150, bbox_inches="tight")
fold_ic_df.to_csv(f"{RESULTS_DIR}/tables/xgboost_per_fold_ic.csv", index=False)
plt.show()